In [1]:
import ee

ee.Authenticate()
ee.Initialize(project='mineral-prospectivity-zim')
print("GEE initialized successfully")

GEE initialized successfully


In [2]:
s2 = ee.Image('projects/mineral-prospectivity-zim/assets/sentinel2_Zimbabwe_dry_season')
landsat = ee.Image('projects/mineral-prospectivity-zim/assets/landsat_Zimbabwe_dry_season')

print("S2 bands:", s2.bandNames().getInfo())
print("Landsat bands:", landsat.bandNames().getInfo())

S2 bands: ['Blue', 'Green', 'Red', 'NIR', 'RE1', 'RE2', 'RE3', 'RE4', 'SWIR1', 'SWIR2']
Landsat bands: ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'TIR']


In [3]:
# ====== Simple ratio indices (classic Crozier / USGS style) ======
s2_ioi   = s2.select('Red').divide(s2.select('Blue')).rename('S2_IOI')          # Ferric iron
s2_fii   = s2.select('SWIR1').divide(s2.select('RE4')).rename('S2_FII')          # Ferrous iron (red‑edge variant)
s2_ci    = s2.select('SWIR1').divide(s2.select('SWIR2')).rename('S2_CI')         # Clay (Al‑OH)
s2_carbi = s2.select('SWIR2').divide(s2.select('SWIR1')).rename('S2_CarbI')     # Carbonate (CO₃)

# ====== Normalised‑difference versions (additional features) ======
s2_ioi_nd = s2.normalizedDifference(['Red', 'Blue']).rename('S2_IOI_nd')
s2_fii_nd = s2.normalizedDifference(['SWIR1', 'RE4']).rename('S2_FII_nd')
s2_ci_nd  = s2.normalizedDifference(['SWIR1', 'SWIR2']).rename('S2_CI_nd')
# Carbonate ND would be exactly -CI_nd; omitted to avoid perfect collinearity.

# ====== Vegetation / moisture (already normalised differences) ======
s2_ndvi = s2.normalizedDifference(['NIR', 'Red']).rename('S2_NDVI')
s2_ndwi = s2.normalizedDifference(['Green', 'NIR']).rename('S2_NDWI')

# Stack all S2 indices into one image
s2_indices = ee.Image.cat([
    s2_ioi, s2_fii, s2_ci, s2_carbi,          # ratio indices
    s2_ioi_nd, s2_fii_nd, s2_ci_nd,           # ND indices
    s2_ndvi, s2_ndwi                          # veg/moisture
])

print("S2 indices computed:", s2_indices.bandNames().getInfo())
# Should show 9 bands: S2_IOI, S2_FII, S2_CI, S2_CarbI, S2_IOI_nd, S2_FII_nd, S2_CI_nd, S2_NDVI, S2_NDWI

S2 indices computed: ['S2_IOI', 'S2_FII', 'S2_CI', 'S2_CarbI', 'S2_IOI_nd', 'S2_FII_nd', 'S2_CI_nd', 'S2_NDVI', 'S2_NDWI']


In [4]:
# ====== Simple ratio indices (standard Landsat band mapping) ======
l8_ioi   = landsat.select('Red').divide(landsat.select('Blue')).rename('L8_IOI')
l8_fii   = landsat.select('SWIR1').divide(landsat.select('NIR')).rename('L8_FII')      # SWIR1/NIR for L8
l8_ci    = landsat.select('SWIR1').divide(landsat.select('SWIR2')).rename('L8_CI')
l8_carbi = landsat.select('SWIR2').divide(landsat.select('SWIR1')).rename('L8_CarbI')

# ====== Normalised‑difference versions ======
l8_ioi_nd = landsat.normalizedDifference(['Red', 'Blue']).rename('L8_IOI_nd')
l8_fii_nd = landsat.normalizedDifference(['SWIR1', 'NIR']).rename('L8_FII_nd')
l8_ci_nd  = landsat.normalizedDifference(['SWIR1', 'SWIR2']).rename('L8_CI_nd')

# ====== Vegetation / moisture ======
l8_ndvi = landsat.normalizedDifference(['NIR', 'Red']).rename('L8_NDVI')
l8_ndwi = landsat.normalizedDifference(['Green', 'NIR']).rename('L8_NDWI')

l8_indices = ee.Image.cat([
    l8_ioi, l8_fii, l8_ci, l8_carbi,
    l8_ioi_nd, l8_fii_nd, l8_ci_nd,
    l8_ndvi, l8_ndwi
])

print("L8 indices computed:", l8_indices.bandNames().getInfo())
# Should show 9 bands: L8_IOI, L8_FII, L8_CI, L8_CarbI, L8_IOI_nd, L8_FII_nd, L8_CI_nd, L8_NDVI, L8_NDWI

L8 indices computed: ['L8_IOI', 'L8_FII', 'L8_CI', 'L8_CarbI', 'L8_IOI_nd', 'L8_FII_nd', 'L8_CI_nd', 'L8_NDVI', 'L8_NDWI']


In [5]:
# Landsat raw bands currently have the same names as Sentinel‑2 raw bands.
# To avoid Earth Engine automatically adding "_1" when concatenating,
# we rename them with a clear prefix.

landsat_renamed = landsat.select(
    ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'TIR'],
    ['L8_Blue', 'L8_Green', 'L8_Red', 'L8_NIR', 'L8_SWIR1', 'L8_SWIR2', 'L8_TIR']
)

print("Landsat renamed bands:", landsat_renamed.bandNames().getInfo())
# Output: ['L8_Blue', 'L8_Green', 'L8_Red', 'L8_NIR', 'L8_SWIR1', 'L8_SWIR2', 'L8_TIR']

Landsat renamed bands: ['L8_Blue', 'L8_Green', 'L8_Red', 'L8_NIR', 'L8_SWIR1', 'L8_SWIR2', 'L8_TIR']


In [6]:
feature_stack = ee.Image.cat([
    s2_indices,       # 9 bands (S2 indices)
    s2,               # 10 bands (S2 raw: Blue, Green, Red, NIR, RE1, RE2, RE3, RE4, SWIR1, SWIR2)
    l8_indices,       # 9 bands (L8 indices)
    landsat_renamed   # 7 bands (L8 raw: L8_Blue, L8_Green, L8_Red, L8_NIR, L8_SWIR1, L8_SWIR2, L8_TIR)
])

print("Total feature bands:", feature_stack.bandNames().getInfo())

# Expected: 9 + 10 + 9 + 7 = 35 bands, all with unique names.

Total feature bands: ['S2_IOI', 'S2_FII', 'S2_CI', 'S2_CarbI', 'S2_IOI_nd', 'S2_FII_nd', 'S2_CI_nd', 'S2_NDVI', 'S2_NDWI', 'Blue', 'Green', 'Red', 'NIR', 'RE1', 'RE2', 'RE3', 'RE4', 'SWIR1', 'SWIR2', 'L8_IOI', 'L8_FII', 'L8_CI', 'L8_CarbI', 'L8_IOI_nd', 'L8_FII_nd', 'L8_CI_nd', 'L8_NDVI', 'L8_NDWI', 'L8_Blue', 'L8_Green', 'L8_Red', 'L8_NIR', 'L8_SWIR1', 'L8_SWIR2', 'L8_TIR']


In [7]:
zimbabwe = ee.Geometry.BBox(25.0, -22.5, 33.5, -15.5)

stats = feature_stack.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=zimbabwe,
    scale=300,
    maxPixels=1e13,
    tileScale=4,
    bestEffort=True
)

print("Feature stack mean values:", stats.getInfo())
# Look for any band with null, negative extreme, or unrealistic values.

Feature stack mean values: {'Blue': 0.06508890592810981, 'Green': 0.08821220619817252, 'L8_Blue': 0.05367879924192432, 'L8_CI': 1.4279730616354613, 'L8_CI_nd': 0.17376149790788603, 'L8_CarbI': 0.7064492469896825, 'L8_FII': 1.2650486831162138, 'L8_FII_nd': 0.11079138728455816, 'L8_Green': 0.0835255276631737, 'L8_IOI': 2.173110338207759, 'L8_IOI_nd': 0.3600104011696505, 'L8_NDVI': 0.3402673385506204, 'L8_NDWI': -0.46560604398223315, 'L8_NIR': 0.22992996759351322, 'L8_Red': 0.11396191461694284, 'L8_SWIR1': 0.29163864716501836, 'L8_SWIR2': 0.20751416699719255, 'L8_TIR': 305.3974479476871, 'NIR': 0.22404491516174424, 'RE1': 0.15539216042761472, 'RE2': 0.18771847461194494, 'RE3': 0.20874733283437583, 'RE4': 0.23765314891105457, 'Red': 0.12336142885703616, 'S2_CI': 1.3731527498578036, 'S2_CI_nd': 0.1549380523268571, 'S2_CarbI': 0.7340652563962128, 'S2_FII': 1.296659419725637, 'S2_FII_nd': 0.12223379398955639, 'S2_IOI': 1.902512127673665, 'S2_IOI_nd': 0.3047472603354431, 'S2_NDVI': 0.293206889

In [8]:
task = ee.batch.Export.image.toAsset(
    image=feature_stack,
    description='feature_stack_export',
    assetId='projects/mineral-prospectivity-zim/assets/feature_stack',
    region=zimbabwe,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e13
)

task.start()
print("Export task started. Task ID:", task.id)

Export task started. Task ID: GWHBEI7EFUITSETOHNJHAYR4
